# Get tcBF and acBF STEM image with GPU

**Before running the notebook, making sure to install the source code following the instruction in GitHub repo: https://github.com/chiahao3/fast-acbf**

> Note: This notebook is designed to demonstrate how to get tilt-corrected and aberration-corrected Bright-field STEM (tcBF and acBF) images from 4D-STEM data

> Note: The comments regarding timing were all using the Moire phason tBL-WSe2 dataset as a benchmark on my 5000 Ada GPU with upscale=1, full_gpu, max_order=2

Author: 
- Dr. Chia-Hao Lee , chia-hao.lee@cornell.edu

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
work_dir = "./" # Change the working directory if needed, but by default it's at the root level of this repo
os.chdir(work_dir)
print("Current working dir: ", os.getcwd())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from ptyrad.optics.constants import get_wavelength_ang
from ptyrad.io.generic import load_raw
from ptyrad.io.handlers import load_array_from_file
from ptyrad.utils.image_proc import guess_radius_of_bright_field_disk
from ptyrad.runtime.diagnostics import print_system_info
from fast_acbf import BFSolver
from tifffile import imwrite

print_system_info()

# Initialize dataset

In [ ]:
# tBL-WSe2 - Moire Phason paper
# file_path = '/home/cl2696/scratch/Figure 4/scan_x128_y128.raw'

# dataset = load_raw(file_path, shape=(16384,128,128))
# dataset = dataset.reshape(128,128,128,128)
# mean_DP = dataset.mean((0,1))
# dataset /= mean_DP.max() # Normalize such that PACBED has max intensity ~ 1

# kv = 80
# max_alpha = 25 # mrad
# scan_step_size = 0.43 # Ang
# Npix = 128
# dk = max_alpha*1e-3 / get_wavelength_ang(kv) / guess_radius_of_bright_field_disk(dataset.mean(axis=(0,1)), thresh=0.5) # Ang-1
# aberrations = {'C10': 80} # Ang, C10 = -defocus, positive C10 is overfocus. Note that we can let AD refine these values later.
# coord_transform = {'flipud': True}

# ====================================================

# tBL-WSe2 - Uncorrected ptycho
# file_path = '/home/cl2696/scratch/tBL_WSe2/Panel_g-h_Themis/scan_x128_y128.raw'

# dataset = load_raw(file_path, shape=(16384,128,128))
# dataset = dataset.reshape(128,128,128,128)
# mean_DP = dataset.mean((0,1))
# dataset /= mean_DP.max() # Normalize such that PACBED has max intensity ~ 1

# kv = 80
# max_alpha = 25 # mrad
# scan_step_size = 0.43 # Ang
# Npix = 128
# dk = max_alpha*1e-3 / get_wavelength_ang(kv) / guess_radius_of_bright_field_disk(dataset.mean(axis=(0,1)), thresh=0.5) # Ang-1
# aberrations = {'C10': 0} # Ang, C10 = -defocus, positive C10 is overfocus. Note that we can let AD refine these values later.
# coord_transform = {'flipud': True}


# ====================================================

# MOSS6
# file_path = '/home/cl2696/scratch/MOSS6/ExperimentalData_MOSS-6_Fig.3/scan_x256_y256.raw'

# dataset = load_raw(file_path, shape=(65536,128,128))
# dataset = dataset.reshape(256,256,128,128)
# dataset[dataset < 20] = 0 # Clip below 20
# mean_DP = dataset.mean((0,1))
# dataset /= mean_DP.max() # Normalize such that PACBED has max intensity ~ 1

# kv = 300
# max_alpha = 10 # mrad
# scan_step_size = 1.051 # Ang
# Npix = 128
# dk = max_alpha*1e-3 / get_wavelength_ang(kv) / guess_radius_of_bright_field_disk(dataset.mean(axis=(0,1)), thresh=0.5) # Ang-1
# aberrations = {'C10': -885} # Ang, C10 = -defocus, positive C10 is overfocus. Note that we can let AD refine these values later.
# coord_transform = {'flipud': True}

# ====================================================

# PSO
file_path = '/home/cl2696/scratch/PSO/60_PSO_area9_41Mx_scan1_CL770mm_CA50um_21p4mrad_spot9_mono21p51_30pA_df20nm_50x_50y_100z_432step_x128_y128.raw'

dataset = load_raw(file_path, shape=(16384,128,128))
dataset = dataset.reshape(128,128,128,128)
mean_DP = dataset.mean((0,1))
dataset /= mean_DP.max() # Normalize such that PACBED has max intensity ~ 1

kv = 300
max_alpha = 21.4 # mrad
scan_step_size = 0.41 # Ang
Npix = 128
dk = max_alpha*1e-3 / get_wavelength_ang(kv) / guess_radius_of_bright_field_disk(dataset.mean(axis=(0,1)), thresh=0.5) # Ang-1
aberrations = {'C10': 200} # Ang, C10 = -defocus, positive C10 is overfocus. Note that we can let AD refine these values later.
coord_transform = {'transpose': True}

# ====================================================

# tBL-STO
# file_path = '/home/cl2696/scratch/20240820_STO_bilayer_04_Hari/data_roi1_Ndp256_dp.hdf5'

# dataset = load_array_from_file(file_path, key='dp')
# dataset = dataset.reshape(256,256,128,128)
# mean_DP = dataset.mean((0,1))
# dataset /= mean_DP.max() # Normalize such that PACBED has max intensity ~ 1

# kv = 300
# max_alpha = 30 # mrad
# defocus = -120 # Ang, positive is underfocs
# scan_step_size = 0.38595 # Ang
# Npix = 128
# dk = max_alpha*1e-3 / get_wavelength_ang(kv) / guess_radius_of_bright_field_disk(dataset.mean(axis=(0,1)), thresh=0.5) # Ang-1
# aberrations = {'C10': 120} # Ang, C10 = -defocus, positive C10 is overfocus. Note that we can let AD refine these values later.
# coord_transform = {}

In [ ]:

fig, axs = plt.subplots(1,2, figsize=(12,6))
im0 = axs[0].imshow(dataset.mean(axis=(0,1)))
im1 = axs[1].imshow(dataset[:,:,Npix//2-4:Npix//2+4,Npix//2-4:Npix//2+4].sum(axis=(-2,-1)), cmap='gray')
fig.colorbar(im0, shrink=0.6)
fig.colorbar(im1, shrink=0.6)
plt.show()

# Fit experimental probe

In [ ]:
from ptyrad.io.load import load_ptyrad
from ptyrad.plotting import plot_probe_modes
# model_path = '/home/cl2696/altas/workspace/ptyrad/output/tBL_WSe2/20260420_full_N16384_dp128_flipT100_random32_p6_1obj_6slice_dz2_plr1e-4_oalr5e-4_oplr5e-4_slr5e-4_orblur0.4_oposc_sng1.0_spr0.1_vlong/model_iter4000.hdf5'

model_path = '/home/cl2696/altas/workspace/ptyrad/output/PSO/20260328_full_N4096_dp256_random32_p4_1obj_21slice_dz10_dpblur1.0_orblur0.4_ozblur1.0_mamp0.03_4.0_oathr0.96_oposc_sng1.0_spr0.1/model_iter0200.hdf5'

ckpt = load_ptyrad(model_path)
probe = ckpt['optimizable_tensors']['probe']

In [ ]:
plot_probe_modes(probe)
plot_probe_modes(probe, amp_or_phase='amplitude', real_or_fourier='fourier')
plot_probe_modes(probe, amp_or_phase='phase', real_or_fourier='fourier')

In [ ]:
import torch
from fast_acbf.bf_solver import make_soft_aperture_torch

In [ ]:
# Centered k-space grids (Å⁻¹), shape (Ny, Nx)

Ny, Nx = probe.shape[-2:]
device='cpu'
kv = 300
wavelength = get_wavelength_ang(kv)
rolloff_mrad=5

ky_1d = torch.fft.fftshift(torch.fft.fftfreq(Ny, d=1.0, device=device)) * Ny * dk
kx_1d = torch.fft.fftshift(torch.fft.fftfreq(Nx, d=1.0, device=device)) * Nx * dk
kY_grid, kX_grid = torch.meshgrid(ky_1d, kx_1d, indexing='ij')  # (Ny, Nx)

# Scattering angle and soft-aperture mask
alpha = torch.sqrt(kX_grid ** 2 + kY_grid ** 2) * wavelength
hard_mask = make_soft_aperture_torch(alpha, max_alpha, rolloff_mrad=0)
soft_mask = make_soft_aperture_torch(alpha, max_alpha, rolloff_mrad=rolloff_mrad)

plt.figure()
plt.imshow(soft_mask.detach().cpu().numpy())
plt.show()

plt.figure()
plt.plot(hard_mask[64,:].detach().cpu().numpy())
plt.plot(soft_mask[64,:].detach().cpu().numpy())
plt.show()

In [ ]:
from fast_acbf.bf_solver import ProbeFitter
fitter = ProbeFitter(probe[0:], 
                     max_alpha=max_alpha, 
                     wavelength=get_wavelength_ang(kv), 
                     aberrations={'C10': 200},
                     dk=dk, 
                     max_order=3, 
                     rolloff_mrad=4,
                     device='cuda',
                     optimize_beam_tilt=False,
                     optimize_kspace_amplitude=True,
                     )
fitter.center_aperture()   # shifts k-space disk to (0,0) — mainly for 'complex'/'3d_stack'

fitter.fit(iters=200, lr=0.1, metric='complex')   # 'intensity', or 'amplitude', 'complex', '3d_stack'
fitter.print_aberrations()
fitter.plot_fit()

# fitter.fit(iters=200, metric='3d_stack', dz_range=(-200,200), Nz=21)   # 'intensity', or 'amplitude', 'complex', '3d_stack'
# fitter.print_aberrations()
# fitter.plot_fit()